# Site-level run coverage
How many protected areas (PAs) the site-level global run evaluated, out of the eligible PAs and out of every PA in the WDPA. A PA is one WDPA `SITE_ID`, which can have several polygons.

- **Evaluated**: sites in the merged results in `global_results/`, written by `unchunker.py`. A site is merged only if it has relative results, absolute results, a report card and match diagnostics, all without errors. Match quality is not considered.
- **Eligible**: every site in the chunk table (`chunks/site_chunks.parquet`, written by `chunker.py`): PAs whose Terrestrial or Coastal polygons are Inscribed, Adopted, Designated or Established, are not UNESCO-MAB Biosphere Reserves, have a STATUS_YR from 1 to 2021, and have GIS_AREA summing to at least 100 km².
- **All PAs**: distinct `SITE_ID`s of PAs in the WDPA polygons and points, from the release the run used, whose polygons or points are Terrestrial or Coastal, are Inscribed, Adopted, Designated or Established (the eligible statuses), and are not UNESCO-MAB Biosphere Reserves. There is no STATUS_YR or area filter.

In [1]:
from pathlib import Path
import sys

import ee
import pandas as pd

# Repo root: nearest folder up from here with pyproject.toml
REPO = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
sys.path.insert(0, str(REPO / "src"))

from cells.get_control_cells import init_ee
from sites.chunker import SITE_STATUSES
from sites.unchunker import RESULT_TABLES
from utils.variables import (
    PROJECT,
    PAS_ASSET_ID,
    SITE_REALMS,
    MAB_DESIGNATION,
    SITE_CHUNKS,
    GLOBAL_RESULTS_PREFIX,
)

# WDPA PAs reported as points
WDPA_POINTS_ASSET_ID = "WCMC/WDPA/current/points"

init_ee(PROJECT)

## Evaluated PAs

In [2]:
results = {
    name: pd.read_csv(f"{GLOBAL_RESULTS_PREFIX}{name}.csv") for name in RESULT_TABLES
}
evaluated = set(results["relative_results"]["site_id"])
# Each table has one row per evaluated site
for name, table in results.items():
    assert table["site_id"].is_unique, f"{name} has a site more than once"
    assert set(table["site_id"]) == evaluated, f"{name} has different sites"
n_evaluated = len(evaluated)

## Eligible PAs

In [3]:
chunks = pd.read_parquet(SITE_CHUNKS)
dropped = pd.read_csv(f"{GLOBAL_RESULTS_PREFIX}dropped_sites.csv")
assert chunks["site_id"].is_unique
eligible = set(chunks["site_id"])
# Every eligible site is either evaluated or dropped, not both
assert evaluated <= eligible
assert set(dropped["site_id"]) == eligible - evaluated
n_eligible = len(eligible)

## All PAs in the WDPA
Only the polygons' release is checked against the run's: the points are a separate asset with its own update time.

In [4]:
# Run's WDPA release, from the chunk table
run_release = chunks["wdpa_update_time"].unique()
assert len(run_release) == 1
current_release = ee.data.getAsset(PAS_ASSET_ID)["updateTime"]
assert current_release == run_release[0], (
    f"WDPA is now {current_release}; the run used {run_release[0]}"
)

# Terrestrial and Coastal PA polygons and points with an eligible status, not UNESCO-MAB
pas = (
    ee.FeatureCollection(PAS_ASSET_ID)
    .merge(ee.FeatureCollection(WDPA_POINTS_ASSET_ID))
    .filter(ee.Filter.eq("SITE_TYPE", "PA"))
    .filter(ee.Filter.inList("REALM", SITE_REALMS))
    .filter(ee.Filter.inList("STATUS", SITE_STATUSES))
    .filter(ee.Filter.neq("DESIG_ENG", MAB_DESIGNATION))
)
n_world = pas.aggregate_array("SITE_ID").distinct().size().getInfo()

## Summary

In [5]:
print(f"WDPA release: {run_release[0]}")
print(f"PAs evaluated (with recorded results): {n_evaluated:,}")
print(f"Eligible PAs we tried to evaluate:     {n_eligible:,}")
print(f"Share of eligible PAs evaluated:       {n_evaluated / n_eligible:.1%}")
print(f"All PAs in the WDPA:                   {n_world:,}")
print(f"Share of all PAs evaluated:            {n_evaluated / n_world:.2%}")

WDPA release: 2026-11-06T01:56:48.667983Z
PAs evaluated (with recorded results): 12,759
Eligible PAs we tried to evaluate:     13,912
Share of eligible PAs evaluated:       91.7%
All PAs in the WDPA:                   477,153
Share of all PAs evaluated:            2.67%
